In [1]:
import pandas as pd
import numpy as np
import pickle

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedShuffleSplit, RandomizedSearchCV
from sklearn.metrics import (
    precision_score,
    recall_score,
    roc_auc_score,
    accuracy_score,
    f1_score
)

from imblearn.over_sampling import RandomOverSampler

In [2]:
seed = 42
n_splits = 5

# Stratified splits
kf = StratifiedShuffleSplit(
    n_splits=n_splits,
    test_size=1 / n_splits,
    random_state=seed
)

metric_list = ["precision", "recall", "roc_auc", "accuracy", "f1"]

# Hyperparameter
param_grid = {
    "max_depth": [None, 5, 10, 15, 20],
    "max_features": ["sqrt", "log2"],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4, 8]
}

In [3]:
def evaluate_random_forest(model_number, upsample=False):
    """
    Evaluate RF using hyperparameter tuning, cross-validation and test set.
    """

    # Load training and test data
    X_train_full = pd.read_csv(
        f"../data/X_train_{model_number}.csv",
        keep_default_na=False
    )

    y_train_full = pd.read_csv(
        f"../data/y_train_{model_number}.csv",
        keep_default_na=False
    ).values.ravel()

    X_test = pd.read_csv(
        f"../data/X_test_{model_number}.csv",
        keep_default_na=False
    )

    y_test = pd.read_csv(
        f"../data/y_test_{model_number}.csv",
        keep_default_na=False
    ).values.ravel()

    # Check
    print("\n")
    print(model_number)

    print("X_train shape:", X_train_full.shape)
    print("X_test shape:", X_test.shape)

    print("\ny_train distribution:")
    print(pd.Series(y_train_full).value_counts())

    print("\ny_test distribution:")
    print(pd.Series(y_test).value_counts())

    # Prepare data
    if upsample:
        upsampler = RandomOverSampler(random_state=seed)
        X_tune, y_tune = upsampler.fit_resample(
            X_train_full,
            y_train_full
        )
    else:
        X_tune = X_train_full
        y_tune = y_train_full

    # Create a base Random Forest model
    base_rf = RandomForestClassifier(
        n_estimators=250,
        random_state=seed,
        n_jobs=-1
    )

    # Search for better hyperparameters
    random_search = RandomizedSearchCV(
        estimator=base_rf,
        param_distributions=param_grid,
        n_iter=30,
        scoring="roc_auc",
        cv=kf,
        random_state=seed,
        n_jobs=-1,
        verbose=1
    )

    random_search.fit(X_tune, y_tune)

    best_params = random_search.best_params_

    print("\nBest parameters:")
    print(best_params)

    print("\nBest tuning ROC AUC:")
    print(round(random_search.best_score_, 4))

    # Store cross-validation results
    cv_scores = {
        "fold": [],
        "cv_precision": [],
        "cv_recall": [],
        "cv_roc_auc": [],
        "cv_accuracy": [],
        "cv_f1": []
    }

    # Run cv using the best parameters
    splits = list(kf.split(X_train_full, y_train_full))

    for fold, (train_idx, val_idx) in enumerate(splits, start=1):

        # Split data into training and validation
        X_train = X_train_full.iloc[train_idx]
        y_train = y_train_full[train_idx]

        X_val = X_train_full.iloc[val_idx]
        y_val = y_train_full[val_idx]

        # Upsampling to the training
        if upsample:
            upsampler = RandomOverSampler(random_state=seed)
            X_train, y_train = upsampler.fit_resample(
                X_train,
                y_train
            )

        # Train RF with the selected parameters
        clf = RandomForestClassifier(
            **best_params,
            random_state=seed,
            n_jobs=-1
        )

        clf.fit(X_train, y_train)

        # Predict labels and probabilities
        preds = clf.predict(X_val)
        prob_preds = clf.predict_proba(X_val)[:, 1]

        # Save scores
        cv_scores["fold"].append(fold)
        cv_scores["cv_precision"].append(precision_score(y_val, preds, zero_division=0))
        cv_scores["cv_recall"].append(recall_score(y_val, preds, zero_division=0))
        cv_scores["cv_roc_auc"].append(roc_auc_score(y_val, prob_preds))
        cv_scores["cv_accuracy"].append(accuracy_score(y_val, preds))
        cv_scores["cv_f1"].append(f1_score(y_val, preds, zero_division=0))

    cv_results = pd.DataFrame(cv_scores)
    cv_summary = cv_results.drop(columns=["fold"]).agg(["mean", "sem"])

    print("\nCross-validation results:")
    display(cv_results)

    print("\nCross-validation summary:")
    display(cv_summary)

    if upsample:
        upsampler = RandomOverSampler(random_state=seed)
        X_train_final, y_train_final = upsampler.fit_resample(
            X_train_full,
            y_train_full
        )
    else:
        X_train_final = X_train_full
        y_train_final = y_train_full

    # Train the final model on the full training set
    final_model = RandomForestClassifier(
        **best_params,
        random_state=seed,
        n_jobs=-1
    )

    final_model.fit(X_train_final, y_train_final)

    # Evaluate the final model on test set
    test_preds = final_model.predict(X_test)
    test_prob_preds = final_model.predict_proba(X_test)[:, 1]
    test_results = pd.DataFrame([{
        "test_precision": precision_score(y_test, test_preds, zero_division=0),
        "test_recall": recall_score(y_test, test_preds, zero_division=0),
        "test_roc_auc": roc_auc_score(y_test, test_prob_preds),
        "test_accuracy": accuracy_score(y_test, test_preds),
        "test_f1": f1_score(y_test, test_preds, zero_division=0)
    }])


    print("\nFinal test results:")
    display(test_results)

    return (
        best_params,
        cv_results,
        cv_summary,
        test_results,
        final_model
    )

In [4]:
# Run
model_settings = {
    "model_1": False,
    "model_1a": True,
    "model_1b": True,
    "model_2": False,
    "model_2a": True,
    "model_2b": True
}

best_params_dict = {}
cv_results_dict = {}
cv_summary_dict = {}
test_results_list = []
final_model_dict = {}

for model_name, use_upsample in model_settings.items():

    (
        best_params,
        cv_results,
        cv_summary,
        test_results,
        final_model
    ) = evaluate_random_forest(
        model_name,
        upsample=use_upsample
    )

    best_params_dict[model_name] = best_params
    cv_results_dict[model_name] = cv_results
    cv_summary_dict[model_name] = cv_summary
    final_model_dict[model_name] = final_model

    test_results["model"] = model_name
    test_results_list.append(test_results)



model_1
X_train shape: (32108, 59)
X_test shape: (8028, 59)

y_train distribution:
1    17321
0    14787
Name: count, dtype: int64

y_test distribution:
1    4337
0    3691
Name: count, dtype: int64
Fitting 5 folds for each of 30 candidates, totalling 150 fits

Best parameters:
{'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 20}

Best tuning ROC AUC:
0.9085

Cross-validation results:


,fold,cv_precision,cv_recall,cv_roc_auc,cv_accuracy,cv_f1
0,1,0.843171,0.841224,0.902716,0.829960,0.842197
1,2,0.841527,0.840069,0.904768,0.828402,0.840797
2,3,0.851863,0.844977,0.908586,0.837122,0.848406
3,4,0.840334,0.843245,0.905287,0.829025,0.841787
4,5,0.856178,0.842090,0.911729,0.838524,0.849076



Cross-validation summary:


,cv_precision,cv_recall,cv_roc_auc,cv_accuracy,cv_f1
mean,0.846615,0.842321,0.906617,0.832607,0.844452
sem,0.003132,0.000844,0.001588,0.002155,0.001769



Final test results:


...

,fold,cv_precision,cv_recall,cv_roc_auc,cv_accuracy,cv_f1
0,1,0.834933,0.893224,0.829816,0.794592,0.863095
1,2,0.840271,0.891170,0.836111,0.798313,0.864973
2,3,0.839177,0.892882,0.833477,0.798313,0.865196
3,4,0.831576,0.890486,0.832226,0.789878,0.860023
4,5,0.841265,0.883299,0.828552,0.794592,0.861770



Cross-validation summary:


,cv_precision,cv_recall,cv_roc_auc,cv_accuracy,cv_f1
mean,0.837444,0.890212,0.832036,0.795138,0.863011
sem,0.001821,0.001802,0.001338,0.001556,0.000977



Final test results:


,test_precision,test_recall,test_roc_auc,test_accuracy,test_f1
0,0.826541,0.893939,0.828439,0.78845,0.85892


In [5]:
# Save
cv_results_all = pd.concat(cv_results_dict, names=["model", "row"])
cv_summary_all = pd.concat(cv_summary_dict, names=["model", "stat"])

test_summary_all = pd.concat(test_results_list, ignore_index=True)
test_summary_all = test_summary_all[
    [
        "model",
        "test_precision",
        "test_recall",
        "test_roc_auc",
        "test_accuracy",
        "test_f1"
    ]
]

best_params_all = pd.DataFrame(best_params_dict).T

print("Best parameters:")
display(best_params_all)

print("\nCV summaries:")
display(cv_summary_all)

print("\nTest results:")
display(test_summary_all)

best_params_all.to_csv("../data/random_forest_best_params_all.csv")
cv_results_all.to_csv("../data/random_forest_cv_results_all.csv")
cv_summary_all.to_csv("../data/random_forest_cv_summary_all.csv")
test_summary_all.to_csv("../data/random_forest_test_summary_all.csv", index=False)

with open("../data/random_forest_final_models.pkl", "wb") as f:
    pickle.dump(final_model_dict, f)

Best parameters:


,min_samples_split,min_samples_leaf,max_features,max_depth
model_1,2,1,sqrt,20
model_1a,2,1,log2,None
model_1b,2,1,log2,None
model_2,2,1,sqrt,20
model_2a,2,1,log2,None
model_2b,2,1,log2,None



CV summaries:


cv_precision  cv_recall  cv_roc_auc  cv_accuracy     cv_f1
model    stat                                                            
model_1  mean      0.846615   0.842321    0.906617     0.832607  0.844452
         sem       0.003132   0.000844    0.001588     0.002155  0.001769
model_1a mean      0.687015   0.509740    0.847610     0.813963  0.585198
         sem       0.010216   0.009740    0.004022     0.004111  0.009838
model_1b mean      0.865643   0.909091    0.878414     0.835971  0.886802
         sem       0.002697   0.003261    0.001815     0.001855  0.001278
model_2  mean      0.786287   0.879283    0.829394     0.765587  0.830188
         sem       0.001486   0.001012    0.001237     0.001828  0.001256
model_2a mean      0.710836   0.718987    0.771490     0.696823  0.714828
         sem       0.005771   0.004076    0.006027     0.004731  0.003774
model_2b mean      0.837444   0.890212    0.832036     0.795138  0.863011
         sem       0.001821   0.001802    0.001338     0.001556  0.000977


Test results:


,model,test_precision,test_recall,test_roc_auc,test_accuracy,test_f1
0,model_1,0.843368,0.842979,0.905470,0.830593,0.843173
1,model_1a,0.691176,0.541613,0.851169,0.816996,0.607322
2,model_1b,0.869436,0.906355,0.876682,0.837865,0.887512
3,model_2,0.782145,0.883806,0.826988,0.764200,0.829873
4,model_2a,0.721503,0.734630,0.782765,0.707260,0.728007
5,model_2b,0.826541,0.893939,0.828439,0.788450,0.858920
